# Notebook 01: Data Preprocessing
## Wearable-Enhanced Insurance Underwriting — NHANES 2003-2004

**Author:** Lubaba Hassan | 22097014D | PolyU Data Science and Analytics

This notebook:
1. Loads all NHANES 2003-2004 files
2. Merges them on the common participant key (SEQN)
3. Constructs the binary underwriting risk label
4. Engineers features for all three scenarios
5. Saves three clean CSVs ready for modelling

**Label Definition:**
- High Risk (1): Cardiovascular disease diagnosis OR diabetes OR hypertension
- Standard Risk (0): None of the above

**Feature Scenarios:**
- Scenario A: Traditional clinical features only
- Scenario B: Traditional + wearable-derived activity features
- Scenario C: Wearable-derived features only

## 1. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

# Paths
DATA_PATH = '../data/processed/'
OUTPUT_PATH = '../data/processed/'
FIGURES_PATH = '../reports/figures/'

# Create figures directory if it doesn't exist
os.makedirs(FIGURES_PATH, exist_ok=True)

print('All imports successful')
print(f'Data path: {os.path.abspath(DATA_PATH)}')

## 2. Load All NHANES Files

In [ ]:
# Load each NHANES component file
print('Loading NHANES files...')

demo   = pd.read_csv(DATA_PATH + 'DEMO_C.csv')    # Demographics
bmx    = pd.read_csv(DATA_PATH + 'BMX_C.csv')     # Body measurements
bpx    = pd.read_csv(DATA_PATH + 'BPX_C.csv')     # Blood pressure (measured)
smq    = pd.read_csv(DATA_PATH + 'SMQ_C.csv')     # Smoking questionnaire
paq    = pd.read_csv(DATA_PATH + 'PAQ_C.csv')     # Physical activity questionnaire
paqiaf = pd.read_csv(DATA_PATH + 'PAQIAF_C.csv')  # Individual activity METs
mcq    = pd.read_csv(DATA_PATH + 'MCQ_C.csv')     # Medical conditions
diq    = pd.read_csv(DATA_PATH + 'DIQ_C.csv')     # Diabetes
bpq    = pd.read_csv(DATA_PATH + 'BPQ_C.csv')     # Blood pressure questionnaire
hsq    = pd.read_csv(DATA_PATH + 'HSQ_C.csv')     # Health status
l13    = pd.read_csv(DATA_PATH + 'L13_C.csv')     # Total cholesterol + HDL
l13am  = pd.read_csv(DATA_PATH + 'L13AM_C.csv')   # LDL + triglycerides (fasting)

print('Files loaded successfully:')
files = {
    'DEMO': demo, 'BMX': bmx, 'BPX': bpx, 'SMQ': smq,
    'PAQ': paq, 'PAQIAF': paqiaf, 'MCQ': mcq, 'DIQ': diq,
    'BPQ': bpq, 'HSQ': hsq, 'L13': l13, 'L13AM': l13am
}
for name, df in files.items():
    print(f'  {name}: {df.shape[0]:,} rows, {df.shape[1]} columns')

## 3. Engineer Wearable Features from PAQIAF

PAQIAF contains individual activity records with MET values per participant.
We aggregate to one row per participant to create wearable-proxy features.

**PADMETS** = MET value for the activity (metabolic equivalent)
**PADLEVEL** = 1 (moderate) or 2 (vigorous)
**PADTIMES** = times per week
**PADDURAT** = duration in minutes

In [ ]:
# Inspect PAQIAF
print('PAQIAF columns:', paqiaf.columns.tolist())
print('\nSample:')
print(paqiaf.head())
print('\nPADMETS stats:')
print(paqiaf['PADMETS'].describe())
print('\nPADLEVEL distribution:')
print(paqiaf['PADLEVEL'].value_counts())

In [ ]:
# Aggregate PAQIAF to one row per participant
# These become your wearable-derived features

wearable_features = paqiaf.groupby('SEQN').agg(
    total_activities    = ('PADACTIV', 'count'),           # Total number of activities reported
    mean_met            = ('PADMETS', 'mean'),             # Average MET intensity
    total_met_hours     = ('PADMETS', 'sum'),              # Total MET-hours (proxy for weekly activity)
    moderate_count      = ('PADLEVEL', lambda x: (x == 1.0).sum()),  # Moderate activity sessions
    vigorous_count      = ('PADLEVEL', lambda x: (x == 2.0).sum()),  # Vigorous activity sessions
).reset_index()

# Vigorous activity ratio (proportion of sessions that are vigorous)
wearable_features['vigorous_ratio'] = (
    wearable_features['vigorous_count'] / 
    wearable_features['total_activities'].replace(0, np.nan)
)

# Activity level category based on total MET hours
# Based on WHO physical activity guidelines:
# <7.5 MET-hours = Insufficient, 7.5-15 = Moderate, >15 = High
wearable_features['activity_category'] = pd.cut(
    wearable_features['total_met_hours'],
    bins=[0, 7.5, 15, float('inf')],
    labels=[0, 1, 2],  # 0=Insufficient, 1=Moderate, 2=High
    include_lowest=True
).astype(float)

print(f'Wearable features created for {len(wearable_features):,} participants')
print('\nWearable feature summary:')
print(wearable_features.describe())

## 4. Build Traditional Features

Extract and rename key variables from each NHANES file.

In [ ]:
# ── DEMOGRAPHICS ──────────────────────────────────────────────
# RIAGENDR: 1=Male, 2=Female
# RIDAGEYR: Age in years
# RIDRETH1: Race/ethnicity
# INDFMINC: Family income

demo_clean = demo[['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'INDFMINC']].copy()
demo_clean.columns = ['SEQN', 'gender', 'age', 'ethnicity', 'income_cat']

# Binary gender: 1=Male, 0=Female
demo_clean['male'] = (demo_clean['gender'] == 1).astype(int)

print('Demographics loaded:', demo_clean.shape)
print('Age range:', demo_clean['age'].min(), '-', demo_clean['age'].max())
print('Gender distribution:')
print(demo_clean['male'].value_counts())

In [ ]:
# ── BODY MEASUREMENTS ─────────────────────────────────────────
# BMXBMI: Body mass index
# BMXWT: Weight (kg)
# BMXHT: Height (cm)
# BMXWAIST: Waist circumference (cm)

bmx_clean = bmx[['SEQN', 'BMXBMI', 'BMXWT', 'BMXHT', 'BMXWAIST']].copy()
bmx_clean.columns = ['SEQN', 'bmi', 'weight_kg', 'height_cm', 'waist_cm']

# BMI category (standard WHO thresholds)
bmx_clean['bmi_cat'] = pd.cut(
    bmx_clean['bmi'],
    bins=[0, 18.5, 25, 30, float('inf')],
    labels=[0, 1, 2, 3],  # 0=Underweight, 1=Normal, 2=Overweight, 3=Obese
    include_lowest=True
).astype(float)

print('Body measurements loaded:', bmx_clean.shape)
print('BMI stats:')
print(bmx_clean['bmi'].describe())

In [ ]:
# ── BLOOD PRESSURE (MEASURED) ─────────────────────────────────
# Average across up to 3 readings for more reliable measurement
# BPXSY1/2/3: Systolic readings
# BPXDI1/2/3: Diastolic readings

bpx_clean = bpx[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3',
                  'BPXDI1', 'BPXDI2', 'BPXDI3', 'BPXPLS']].copy()

# Average systolic and diastolic across readings
bpx_clean['systolic_bp'] = bpx_clean[['BPXSY1', 'BPXSY2', 'BPXSY3']].mean(axis=1)
bpx_clean['diastolic_bp'] = bpx_clean[['BPXDI1', 'BPXDI2', 'BPXDI3']].mean(axis=1)
bpx_clean['pulse_rate'] = bpx_clean['BPXPLS']

# Hypertension by measurement: systolic >= 140 OR diastolic >= 90
bpx_clean['bp_hypertension_measured'] = (
    (bpx_clean['systolic_bp'] >= 140) | 
    (bpx_clean['diastolic_bp'] >= 90)
).astype(int)

bpx_clean = bpx_clean[['SEQN', 'systolic_bp', 'diastolic_bp', 
                         'pulse_rate', 'bp_hypertension_measured']]

print('Blood pressure loaded:', bpx_clean.shape)
print('BP stats:')
print(bpx_clean[['systolic_bp', 'diastolic_bp']].describe())

In [ ]:
# ── SMOKING ───────────────────────────────────────────────────
# SMQ020: Smoked at least 100 cigarettes in life
# 1=Yes, 2=No, 7=Refused, 9=Don't know

smq_clean = smq[['SEQN', 'SMQ020']].copy()
smq_clean.columns = ['SEQN', 'smoker_raw']

# Binary: 1=Ever smoker, 0=Never smoker, NaN=Missing
smq_clean['ever_smoker'] = smq_clean['smoker_raw'].map({1.0: 1, 2.0: 0})
smq_clean = smq_clean[['SEQN', 'ever_smoker']]

print('Smoking loaded:', smq_clean.shape)
print('Smoker distribution:')
print(smq_clean['ever_smoker'].value_counts())

In [ ]:
# ── CHOLESTEROL (LAB VALUES) ──────────────────────────────────
# L13: Total cholesterol (LBXTC) and HDL (LBXHDD)
# L13AM: LDL (LBDLDL) and triglycerides (LBXTR) - fasting only

l13_clean = l13[['SEQN', 'LBXTC', 'LBXHDD']].copy()
l13_clean.columns = ['SEQN', 'total_cholesterol', 'hdl_cholesterol']

l13am_clean = l13am[['SEQN', 'LBDLDL', 'LBXTR']].copy()
l13am_clean.columns = ['SEQN', 'ldl_cholesterol', 'triglycerides']

# High cholesterol flags (clinical thresholds)
l13_clean['high_total_chol'] = (l13_clean['total_cholesterol'] > 200).astype(int)
l13_clean['low_hdl'] = (l13_clean['hdl_cholesterol'] < 40).astype(int)

print('Total cholesterol/HDL loaded:', l13_clean.shape)
print('LDL/Triglycerides loaded:', l13am_clean.shape)
print('High cholesterol rate:', l13_clean['high_total_chol'].mean().round(3))

In [ ]:
# ── PHYSICAL ACTIVITY QUESTIONNAIRE ───────────────────────────
# PAQ180: Typical daily activity level
# 1=Mostly sitting, 2=Walking, 3=Standing, 4=Heavy labour
# PAD320: Days walked or bicycled

paq_clean = paq[['SEQN', 'PAQ180', 'PAD320']].copy()
paq_clean.columns = ['SEQN', 'daily_activity_level', 'days_walked']

# Clean out refused/don't know codes (7, 9)
paq_clean['daily_activity_level'] = paq_clean['daily_activity_level'].where(
    paq_clean['daily_activity_level'].isin([1, 2, 3, 4])
)

print('Physical activity questionnaire loaded:', paq_clean.shape)
print('Daily activity distribution:')
print(paq_clean['daily_activity_level'].value_counts())

In [ ]:
# ── HEALTH STATUS ─────────────────────────────────────────────
# HSD010: General health condition
# 1=Excellent, 2=Very good, 3=Good, 4=Fair, 5=Poor

hsq_clean = hsq[['SEQN', 'HSD010']].copy()
hsq_clean.columns = ['SEQN', 'self_rated_health']

# Clean out refused/don't know codes
hsq_clean['self_rated_health'] = hsq_clean['self_rated_health'].where(
    hsq_clean['self_rated_health'].isin([1, 2, 3, 4, 5])
)

print('Health status loaded:', hsq_clean.shape)

## 5. Construct the Underwriting Risk Label — Framingham Risk Score

### Justification

The label is constructed using the **Framingham Risk Score (FRS)**, a validated 10-year cardiovascular disease risk model published by D'Agostino et al. (2008) in *Circulation*. This approach is chosen because:

1. FRS is a clinically validated, peer-reviewed risk tool used in medical and insurance practice
2. Life insurers use validated clinical risk scores to determine premium loading — Wang (2021) confirms cardiovascular conditions, diabetes, and hypertension drive underwriting decisions
3. All required variables (age, gender, cholesterol, BP, diabetes, smoking) are present in NHANES
4. A continuous risk score threshold mirrors actual underwriting practice more accurately than a condition checklist

**Label threshold:**
- FRS >= 20%: High Risk (Label = 1) — triggers loading or specialist referral
- FRS < 20%: Standard Risk (Label = 0) — accepted at standard terms

**Reference:** D'Agostino, R.B. et al. (2008). General cardiovascular risk profile for use in primary care. *Circulation*, 117(6), 743-753.

In [ ]:
# Extract BP treatment status needed for Framingham
# BPQ050A: Currently taking prescribed medicine for hypertension
# 1=Yes, 2=No
bpq_treatment = bpq[['SEQN', 'BPQ050A']].copy()
bpq_treatment.columns = ['SEQN', 'bp_treatment_raw']
bpq_treatment['on_bp_treatment'] = (bpq_treatment['bp_treatment_raw'] == 1.0).astype(int)
bpq_treatment = bpq_treatment[['SEQN', 'on_bp_treatment']]

# Also get high cholesterol told and hypertension told for supplementary use
bpq_extra = bpq[['SEQN', 'BPQ020', 'BPQ080']].copy()
bpq_extra.columns = ['SEQN', 'hypertension_told', 'high_chol_told']
bpq_extra['hypertension_told'] = (bpq_extra['hypertension_told'] == 1.0).astype(int)
bpq_extra['high_chol_told'] = (bpq_extra['high_chol_told'] == 1.0).astype(int)

print('BP treatment status:')
print(bpq_treatment['on_bp_treatment'].value_counts())

In [ ]:
# ── FRAMINGHAM RISK SCORE IMPLEMENTATION ──────────────────────
# Source: D'Agostino et al. (2008) Circulation 117(6):743-753
# Separate Cox proportional hazards models for men and women

def compute_framingham_score(row):
    """
    Compute 10-year Framingham cardiovascular risk score.
    Returns probability (0-1). Returns NaN if required variables missing.
    Variables: age, male, total_cholesterol, hdl_cholesterol,
               systolic_bp, on_bp_treatment, diabetes, ever_smoker
    """
    required = ['age', 'male', 'total_cholesterol', 'hdl_cholesterol',
                'systolic_bp', 'on_bp_treatment', 'diabetes', 'ever_smoker']
    if any(pd.isna(row.get(v)) for v in required):
        return np.nan

    age  = float(row['age'])
    tc   = float(row['total_cholesterol'])
    hdl  = float(row['hdl_cholesterol'])
    sbp  = float(row['systolic_bp'])
    bptx = float(row['on_bp_treatment'])
    dm   = float(row['diabetes'])
    smk  = float(row['ever_smoker'])

    # Clamp age to valid model range
    age = max(30.0, min(74.0, age))

    if row['male'] == 1:
        # Men: coefficients from Table 2, D'Agostino et al. (2008)
        ln_sbp_tx   = np.log(sbp) if bptx == 1 else 0.0
        ln_sbp_notx = np.log(sbp) if bptx == 0 else 0.0
        lp = (3.06117  * np.log(age) +
              1.12370  * np.log(tc)  +
             -0.93263  * np.log(hdl) +
              1.93303  * ln_sbp_tx   +
              1.99881  * ln_sbp_notx +
              0.65451  * dm          +
              0.57367  * smk)
        baseline_survival = 0.88936
        mean_coeff        = 23.9802
    else:
        # Women: coefficients from Table 2, D'Agostino et al. (2008)
        ln_sbp_tx   = np.log(sbp) if bptx == 1 else 0.0
        ln_sbp_notx = np.log(sbp) if bptx == 0 else 0.0
        lp = (2.32888  * np.log(age)        +
              1.20904  * (np.log(age) ** 2)  +
              0.70833  * np.log(tc)          +
             -0.87328  * np.log(hdl)         +
              2.76157  * ln_sbp_tx           +
              2.82263  * ln_sbp_notx         +
              0.52873  * dm                  +
              0.52873  * smk)
        baseline_survival = 0.95012
        mean_coeff        = 26.1931

    risk = 1.0 - (baseline_survival ** np.exp(lp - mean_coeff))
    return float(np.clip(risk, 0.0, 1.0))

print('Framingham function defined.')

In [ ]:
# Assemble Framingham input dataframe
frs_df = demo_clean[['SEQN', 'age', 'male']].copy()
frs_df = frs_df.merge(l13_clean[['SEQN', 'total_cholesterol', 'hdl_cholesterol']],
                      on='SEQN', how='left')
frs_df = frs_df.merge(bpx_clean[['SEQN', 'systolic_bp']], on='SEQN', how='left')
frs_df = frs_df.merge(bpq_treatment, on='SEQN', how='left')
frs_df = frs_df.merge(diq_label, on='SEQN', how='left')
frs_df = frs_df.merge(smq_clean, on='SEQN', how='left')

# Fill binary variables conservatively (0 = no condition/treatment)
frs_df['on_bp_treatment'] = frs_df['on_bp_treatment'].fillna(0)
frs_df['diabetes']        = frs_df['diabetes'].fillna(0)
frs_df['ever_smoker']     = frs_df['ever_smoker'].fillna(0)

# Compute FRS
print('Computing Framingham Risk Scores (this may take 30 seconds)...')
frs_df['framingham_score'] = frs_df.apply(compute_framingham_score, axis=1)
frs_df['framingham_pct']   = frs_df['framingham_score'] * 100

print(f'FRS computed: {frs_df["framingham_score"].notna().sum():,} participants')
print(f'FRS missing:  {frs_df["framingham_score"].isna().sum():,} (insufficient lab data)')
print('\nScore distribution:')
print(frs_df['framingham_pct'].describe().round(1))

In [ ]:
# Apply 20% threshold to create binary underwriting label
# Standard threshold in clinical practice (D'Agostino et al., 2008)
# and consistent with underwriting referral triggers (Wang, 2021)

HIGH_RISK_THRESHOLD = 0.20

frs_df['high_risk'] = (
    frs_df['framingham_score'] >= HIGH_RISK_THRESHOLD
).astype(int)

label_df = frs_df[['SEQN', 'framingham_score', 'framingham_pct', 'high_risk']].dropna(
    subset=['framingham_score']
)

print('LABEL DISTRIBUTION (Framingham-based, threshold=20%):')
print(label_df['high_risk'].value_counts())
print(f'\nHigh risk rate:     {label_df["high_risk"].mean():.1%}')
print(f'Standard risk rate: {(1-label_df["high_risk"]).mean():.1%}')
print(f'\nFRS statistics by label:')
print(label_df.groupby('high_risk')['framingham_pct'].describe().round(1))

In [ ]:
# Visualise Framingham score distribution and label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Score histogram with threshold line
ax = axes[0]
ax.hist(label_df['framingham_pct'].clip(0, 60), bins=50,
        color='#3498db', edgecolor='white', alpha=0.8)
ax.axvline(x=20, color='red', linestyle='--', linewidth=2,
           label='20% threshold (High Risk)')
ax.set_xlabel('10-Year CVD Risk (%)')
ax.set_ylabel('Count')
ax.set_title('Framingham Risk Score Distribution', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Right: Binary label bar chart
ax = axes[1]
counts = label_df['high_risk'].value_counts().sort_index()
bars = ax.bar(['Standard Risk\n(FRS < 20%)', 'High Risk\n(FRS >= 20%)'],
              counts.values,
              color=['#3498db', '#e74c3c'],
              edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{count:,}', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Count')
ax.set_title('Binary Underwriting Label Distribution', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.suptitle('Framingham-Based Underwriting Risk Label\n(D\'Agostino et al., 2008)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_PATH + 'framingham_label.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 6. Merge All Data

In [ ]:
# Start from demographics as the base
df = demo_clean.copy()

# Merge all traditional features
df = df.merge(bmx_clean,   on='SEQN', how='left')
df = df.merge(bpx_clean,   on='SEQN', how='left')
df = df.merge(smq_clean,   on='SEQN', how='left')
df = df.merge(l13_clean,   on='SEQN', how='left')
df = df.merge(l13am_clean, on='SEQN', how='left')
df = df.merge(paq_clean,   on='SEQN', how='left')
df = df.merge(hsq_clean,   on='SEQN', how='left')
df = df.merge(bpq_extra,   on='SEQN', how='left')
df = df.merge(bpq_treatment, on='SEQN', how='left')

# Merge wearable features
df = df.merge(wearable_features, on='SEQN', how='left')

# Merge Framingham label
df = df.merge(label_df[['SEQN', 'framingham_score', 'framingham_pct', 'high_risk']],
              on='SEQN', how='inner')  # inner join: keep only participants with FRS

# Flag who has wearable data
df['has_wearable'] = (~df['total_met_hours'].isna()).astype(int)

print(f'Full merged dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Participants with wearable data: {df["has_wearable"].sum():,} ({df["has_wearable"].mean():.1%})')
print(f'High risk rate: {df["high_risk"].mean():.1%}')

## 7. Data Cleaning

In [ ]:
# Restrict to adults aged 18+ (underwriting is for adults)
df = df[df['age'] >= 18].copy()
print(f'After age filter (18+): {df.shape[0]:,} participants')

# Remove extreme BMI outliers (clinical implausibility)
df = df[(df['bmi'].isna()) | ((df['bmi'] >= 10) & (df['bmi'] <= 80))]
print(f'After BMI outlier removal: {df.shape[0]:,} participants')

# Remove extreme blood pressure outliers
df = df[(df['systolic_bp'].isna()) | 
        ((df['systolic_bp'] >= 70) & (df['systolic_bp'] <= 250))]
print(f'After BP outlier removal: {df.shape[0]:,} participants')

# Remove extreme age outliers (none expected but good practice)
df = df[df['age'] <= 85]
print(f'After age cap (85): {df.shape[0]:,} participants')

print(f'\nFinal dataset size: {df.shape[0]:,} participants')
print(f'Label distribution:')
print(df['high_risk'].value_counts())
print(f'High risk rate: {df["high_risk"].mean():.1%}')

## 8. Missing Value Analysis

In [ ]:
# Check missing values in key columns
key_cols = [
    'age', 'male', 'bmi', 'systolic_bp', 'diastolic_bp',
    'ever_smoker', 'total_cholesterol', 'hdl_cholesterol',
    'ldl_cholesterol', 'triglycerides', 'self_rated_health',
    'daily_activity_level', 'high_chol_told',
    'total_met_hours', 'mean_met', 'vigorous_ratio',
    'moderate_count', 'vigorous_count', 'activity_category',
    'high_risk'
]

missing_summary = pd.DataFrame({
    'missing_count': df[key_cols].isnull().sum(),
    'missing_pct': (df[key_cols].isnull().sum() / len(df) * 100).round(1)
})
print('Missing value summary:')
print(missing_summary.to_string())

In [ ]:
# Visualise missing values
fig, ax = plt.subplots(figsize=(10, 6))

missing_pct = (df[key_cols].isnull().sum() / len(df) * 100).sort_values(ascending=True)
colors = ['#e74c3c' if x > 40 else '#f39c12' if x > 20 else '#2ecc71' 
          for x in missing_pct]

ax.barh(missing_pct.index, missing_pct.values, color=colors)
ax.axvline(x=20, color='orange', linestyle='--', alpha=0.7, label='20% threshold')
ax.axvline(x=40, color='red', linestyle='--', alpha=0.7, label='40% threshold')
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values by Feature')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_PATH + 'missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 9. Add Age Bands and Fairness Subgroups

In [ ]:
# Age bands for fairness analysis
df['age_band'] = pd.cut(
    df['age'],
    bins=[18, 30, 45, 60, 75, 85],
    labels=['18-30', '31-45', '46-60', '61-75', '76-85'],
    include_lowest=True
)

print('Age band distribution:')
print(df['age_band'].value_counts().sort_index())

print('\nHigh risk rate by age band:')
print(df.groupby('age_band')['high_risk'].mean().round(3))

print('\nHigh risk rate by gender (1=Male, 0=Female):')
print(df.groupby('male')['high_risk'].mean().round(3))

## 10. Build Three Scenario Datasets

Each scenario has:
- Different feature sets
- Missing values imputed appropriately
- Saved as a clean CSV ready for modelling

In [ ]:
# Define feature sets for each scenario

TRADITIONAL_FEATURES = [
    'age', 'male',
    'bmi', 'waist_cm',
    'systolic_bp', 'diastolic_bp',
    'total_cholesterol', 'hdl_cholesterol',
    'ever_smoker',
    'self_rated_health',
    'daily_activity_level',
    'high_chol_told',
    'bp_hypertension_measured'
]

WEARABLE_FEATURES = [
    'total_met_hours',
    'mean_met',
    'total_activities',
    'moderate_count',
    'vigorous_count',
    'vigorous_ratio',
    'activity_category'
]

# Metadata columns to keep for fairness analysis
META_COLS = ['SEQN', 'age_band', 'ethnicity', 'income_cat', 'high_risk']

print('Traditional features:', len(TRADITIONAL_FEATURES))
print('Wearable features:', len(WEARABLE_FEATURES))

In [ ]:
from sklearn.impute import SimpleImputer

def prepare_scenario(df, feature_cols, label_col='high_risk', 
                     meta_cols=None, strategy='median'):
    """
    Prepare a clean dataset for a given feature scenario.
    
    - Drops rows where label is missing
    - Imputes missing feature values using specified strategy
    - Returns clean dataframe
    """
    if meta_cols is None:
        meta_cols = ['SEQN', 'high_risk']
    
    # Select relevant columns
    cols = list(set(meta_cols + feature_cols + [label_col]))
    scenario_df = df[cols].copy()
    
    # Drop rows where label is missing
    scenario_df = scenario_df.dropna(subset=[label_col])
    
    # Impute features
    # Median for numeric, most_frequent for categorical
    numeric_features = [f for f in feature_cols 
                        if scenario_df[f].dtype in ['float64', 'int64']]
    
    imputer = SimpleImputer(strategy=strategy)
    scenario_df[numeric_features] = imputer.fit_transform(
        scenario_df[numeric_features]
    )
    
    print(f'  Shape: {scenario_df.shape}')
    print(f'  Label distribution: {scenario_df[label_col].value_counts().to_dict()}')
    print(f'  High risk rate: {scenario_df[label_col].mean():.1%}')
    print(f'  Missing values remaining: {scenario_df[feature_cols].isnull().sum().sum()}')
    
    return scenario_df

In [ ]:
# ── SCENARIO A: Traditional features only ─────────────────────
print('Preparing Scenario A (Traditional features only)...')

scenario_a = prepare_scenario(
    df, 
    feature_cols=TRADITIONAL_FEATURES,
    meta_cols=META_COLS
)

scenario_a.to_csv(OUTPUT_PATH + 'scenario_a.csv', index=False)
print('  Saved: scenario_a.csv')

In [ ]:
# ── SCENARIO B: Traditional + Wearable features ───────────────
# Only participants who have wearable data
print('Preparing Scenario B (Traditional + Wearable features)...')

df_b = df[df['has_wearable'] == 1].copy()
print(f'  Participants with wearable data: {len(df_b):,}')

scenario_b = prepare_scenario(
    df_b,
    feature_cols=TRADITIONAL_FEATURES + WEARABLE_FEATURES,
    meta_cols=META_COLS
)

scenario_b.to_csv(OUTPUT_PATH + 'scenario_b.csv', index=False)
print('  Saved: scenario_b.csv')

In [ ]:
# ── SCENARIO C: Wearable features only ───────────────────────
print('Preparing Scenario C (Wearable features only)...')

scenario_c = prepare_scenario(
    df_b,
    feature_cols=WEARABLE_FEATURES,
    meta_cols=META_COLS
)

scenario_c.to_csv(OUTPUT_PATH + 'scenario_c.csv', index=False)
print('  Saved: scenario_c.csv')

## 11. Summary Statistics

In [ ]:
print('=' * 60)
print('PREPROCESSING COMPLETE — DATASET SUMMARY')
print('=' * 60)
print(f'Total NHANES participants loaded:      {demo.shape[0]:,}')
print(f'After FRS computation and cleaning:    {df.shape[0]:,}')
print()
print(f'Scenario A participants:               {len(scenario_a):,}')
print(f'  Features:                            {len(TRADITIONAL_FEATURES)}')
print(f'  High risk rate:                      {scenario_a["high_risk"].mean():.1%}')
print()
print(f'Scenario B participants:               {len(scenario_b):,}')
print(f'  Features:                            {len(TRADITIONAL_FEATURES) + len(WEARABLE_FEATURES)}')
print(f'  High risk rate:                      {scenario_b["high_risk"].mean():.1%}')
print()
print(f'Scenario C participants:               {len(scenario_c):,}')
print(f'  Features:                            {len(WEARABLE_FEATURES)}')
print(f'  High risk rate:                      {scenario_c["high_risk"].mean():.1%}')
print()
print('Label: Framingham Risk Score >= 20% = High Risk')
print('Reference: D\'Agostino et al. (2008) Circulation 117(6):743-753')
print()
print('Files saved to data/processed/:')
print('  scenario_a.csv')
print('  scenario_b.csv')
print('  scenario_c.csv')

In [ ]:
# Label distribution plot
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

scenarios = [
    (scenario_a, 'Scenario A\n(Traditional)', '#3498db'),
    (scenario_b, 'Scenario B\n(Traditional + Wearable)', '#2ecc71'),
    (scenario_c, 'Scenario C\n(Wearable Only)', '#e74c3c')
]

for ax, (scen_df, title, color) in zip(axes, scenarios):
    counts = scen_df['high_risk'].value_counts().sort_index()
    bars = ax.bar(['Standard Risk\n(0)', 'High Risk\n(1)'], 
                  counts.values, 
                  color=[color + '88', color],
                  edgecolor='white', linewidth=1.5)
    
    # Add count labels
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{count:,}', ha='center', va='bottom', fontweight='bold')
    
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_ylabel('Count')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add percentage annotation
    high_risk_pct = scen_df['high_risk'].mean()
    ax.text(0.98, 0.95, f'High Risk: {high_risk_pct:.1%}',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=10, color='gray')

plt.suptitle('Label Distribution Across Feature Scenarios', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_PATH + 'label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# High risk rate by age band
fig, ax = plt.subplots(figsize=(10, 5))

risk_by_age = df.groupby('age_band')['high_risk'].mean() * 100

bars = ax.bar(risk_by_age.index.astype(str), risk_by_age.values,
              color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6'],
              edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, risk_by_age.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')

ax.set_xlabel('Age Band')
ax.set_ylabel('High Risk Rate (%)')
ax.set_title('High Risk Rate by Age Band\n(Validates Label Clinical Plausibility)',
             fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_PATH + 'risk_by_age_band.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# Wearable feature distributions
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

wearable_plot_cols = [
    ('total_met_hours', 'Total MET-Hours'),
    ('mean_met', 'Mean MET Value'),
    ('total_activities', 'Total Activities'),
    ('moderate_count', 'Moderate Activity Count'),
    ('vigorous_count', 'Vigorous Activity Count'),
    ('vigorous_ratio', 'Vigorous Activity Ratio')
]

df_with_wearable = df[df['has_wearable'] == 1]

for ax, (col, title) in zip(axes.flat, wearable_plot_cols):
    # Plot by risk label
    for label, color, name in [(0, '#3498db', 'Standard Risk'), 
                                 (1, '#e74c3c', 'High Risk')]:
        data = df_with_wearable[df_with_wearable['high_risk'] == label][col].dropna()
        ax.hist(data, bins=30, alpha=0.6, color=color, label=name, density=True)
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Wearable Feature Distributions by Risk Label', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_PATH + 'wearable_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
print('Notebook 01 complete.')
print('Next: Run 02_eda.ipynb for exploratory data analysis')
print('Then: Run 03_scenario_a_models.ipynb to train baseline classifiers')